# Run Flask Webapp + ngrok on Kaggle

Notebook ini untuk menjalankan webapp Flask dari project ini di Kaggle dan membuat URL publik menggunakan ngrok.

In [ ]:
# 1) Clone repository (jika belum ada)
# Ganti URL repo sesuai kebutuhan.
import os
from pathlib import Path

REPO_URL = "https://github.com/Herutriana44/Acute-ischemic-stroke-dataset-image-segmentation.git"
PROJECT_DIR = Path("/kaggle/working/Acute-ischemic-stroke-dataset-image-segmentation")

if not PROJECT_DIR.exists():
    !git clone "$REPO_URL" "$PROJECT_DIR"

os.chdir(PROJECT_DIR)
print("Working dir:", Path.cwd())

In [ ]:
# 2) Install dependencies
!pip install -q -r requirements.txt pyngrok waitress

In [ ]:
# 3) Set ngrok auth token
# Opsi A (direkomendasikan): isi token manual di bawah.
# Opsi B: set ENV di Kaggle Secrets/Environment sebagai NGROK_AUTHTOKEN.

import os
from pyngrok import ngrok

NGROK_AUTHTOKEN = os.getenv("NGROK_AUTHTOKEN", "")
# Contoh: NGROK_AUTHTOKEN = "<isi_token_ngrok_anda>"

if not NGROK_AUTHTOKEN:
    raise ValueError("NGROK_AUTHTOKEN belum diset. Isi variabel atau set env NGROK_AUTHTOKEN.")

ngrok.set_auth_token(NGROK_AUTHTOKEN)
print("ngrok token configured")

In [ ]:
# 4) Jalankan Flask app di background (waitress)
import threading
from waitress import serve

from webapp.app import app

PORT = 5000

def run_app():
    # threads tinggi agar inference upload tetap responsif
    serve(app, host="0.0.0.0", port=PORT, threads=8)

server_thread = threading.Thread(target=run_app, daemon=True)
server_thread.start()

print(f"Flask app started on http://0.0.0.0:{PORT}")

In [ ]:
# 5) Buat public URL ngrok (port forwarding)
from pyngrok import ngrok

# Tutup tunnel lama jika ada
for t in ngrok.get_tunnels():
    ngrok.disconnect(t.public_url)

public_tunnel = ngrok.connect(addr=PORT, proto="http")
print("Public URL:", public_tunnel.public_url)

In [ ]:
# 6) (Opsional) monitoring tunnel
from pyngrok import ngrok

print("Active tunnels:")
for t in ngrok.get_tunnels():
    print(f"- {t.public_url} -> {t.config.get('addr')}")